In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def show(title, img):
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(14, 7))
    plt.title(title)
    plt.axis('off')
    plt.imshow(rgb)
    plt.tight_layout()
    plt.show()


def trim(frame):
    if frame is None or frame.size == 0:
        return frame
    if not np.sum(frame[0]):
        return trim(frame[1:])
    if not np.sum(frame[-1]):
        return trim(frame[0:-1])
    if not np.sum(frame[:, 0]):
        return trim(frame[:, 1:])
    if not np.sum(frame[:, -1]):
        return trim(frame[:, :-1])
    return frame


def stitch_two(imgL, imgR):
    grayL = cv2.cvtColor(imgL, cv2.COLOR_BGR2GRAY)
    grayR = cv2.cvtColor(imgR, cv2.COLOR_BGR2GRAY)

    sift = cv2.SIFT_create()

    kpL, dsL = sift.detectAndCompute(grayL, None)
    kpR, dsR = sift.detectAndCompute(grayR, None)

    if dsL is None or dsR is None:
        print("  ⚠️  No descriptors found, skipping pair.")
        return imgL

    matcher  = cv2.BFMatcher()
    matches  = matcher.knnMatch(dsR, dsL, k=2)

    good = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good.append(m)

    print(f"  ✅ Good matches: {len(good)}")

    MIN_MATCH_COUNT = 10
    if len(good) < MIN_MATCH_COUNT:
        print(f"  ⚠️  Not enough matches ({len(good)}/{MIN_MATCH_COUNT}), skipping pair.")
        return imgL

    src_pts = np.float32([kpR[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kpL[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    M, _ = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    if M is None:
        print("  ⚠️  Homography failed, skipping pair.")
        return imgL

    canvas_width  = imgL.shape[1] + imgR.shape[1]
    canvas_height = max(imgL.shape[0], imgR.shape[0])

    warped = cv2.warpPerspective(imgR, M, (canvas_width, canvas_height))

    # Place imgL on the left side of the canvas
    warped[0:imgL.shape[0], 0:imgL.shape[1]] = imgL

    return warped


# ============================================================
# MAIN — PROCESS EACH SUBFOLDER IN ./data
# ============================================================
data_root = "./data"

subfolders = sorted([
    f for f in os.listdir(data_root)
    if os.path.isdir(os.path.join(data_root, f))
])

if not subfolders:
    print("⚠️  No subfolders found in ./data")
    exit()

print(f"📁 Found {len(subfolders)} subfolder(s): {subfolders}\n")

for folder_name in subfolders:
    folder_path = os.path.join(data_root, folder_name)

    # Collect and sort images (0.jpg, 1.jpg, 2.jpg ... left → right)
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff')
    image_paths = []
    for ext in extensions:
        image_paths.extend(glob.glob(os.path.join(folder_path, ext)))

    # Sort by numeric filename so 0 < 1 < 2 < ...
    image_paths = sorted(image_paths, key=lambda p: int(os.path.splitext(os.path.basename(p))[0]))

    print(f"📂 Folder: '{folder_name}' → {len(image_paths)} image(s) found")
    for p in image_paths:
        print(f"    {os.path.basename(p)}")

    if len(image_paths) < 2:
        print("  ⚠️  Need at least 2 images to stitch, skipping.\n")
        continue

    # Load all images
    images = []
    for path in image_paths:
        img = cv2.imread(path)
        if img is None:
            print(f"  ⚠️  Could not load {path}, skipping.")
            continue
        images.append(img)

    if len(images) < 2:
        print("  ⚠️  Not enough loadable images, skipping.\n")
        continue

    # Show individual images before stitching
    for i, img in enumerate(images):
        show(f"[{folder_name}] Image {i}", img)

    # Stitch left → right: start with image 0, merge image 1, then 2, etc.
    print(f"\n  🔗 Stitching {len(images)} images left → right...")
    result = images[0]

    for i in range(1, len(images)):
        print(f"  → Stitching image {i-1} + image {i}")
        result = stitch_two(imgL=result, imgR=images[i])
        show(f"[{folder_name}] After stitching image 0–{i}", result)

    # Trim black borders
    result = trim(result)

    # Show and save final result
    show(f"[{folder_name}] ✅ FINAL STITCHED IMAGE", result)

    output_path = os.path.join(data_root, f"{folder_name}_stitched.jpg")
    cv2.imwrite(output_path, result)
    print(f"  💾 Saved → {output_path}\n")

print("✅ All folders processed.")
